In [8]:
# =========================
# 1) Install
# =========================
!pip -q install -U google-genai

In [9]:
# =========================
# 2) Imports
# =========================
from google import genai
from google.genai import types
from google.colab import files

import mimetypes
import re
import os
import getpass


In [ ]:
# =========================
# 3) API key
# =========================
# Option A: paste directly

api_key =  ""

"""
# Option B: secure input in Colab
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Enter GEMINI_API_KEY: ").strip()

client = genai.Client(api_key=api_key)
"""

'\n# Option B: secure input in Colab\napi_key = os.environ.get("GEMINI_API_KEY")\nif not api_key:\n    api_key = getpass.getpass("Enter GEMINI_API_KEY: ").strip()\n\nclient = genai.Client(api_key=api_key)\n'

In [26]:
# =========================
# 4) Model settings
# =========================
MODEL_NAME = "gemini-2.5-flash"
THINKING_BUDGET = 0
MAX_OUTPUT_TOKENS = 170
TEMPERATURE = 0.1

In [12]:
# =========================
# 5) Upload image in Colab
# =========================
uploaded = files.upload()
image_path = next(iter(uploaded.keys()))
print("Loaded:", image_path)

mime_type = mimetypes.guess_type(image_path)[0] or "image/jpeg"

with open(image_path, "rb") as f:
    image_bytes = f.read()

Saving eczema.jpeg to eczema (1).jpeg
Loaded: eczema (1).jpeg


In [21]:
# =========================
# 6) Sample patient data
# In production, AWS will send the image bytes + patient info.
# =========================
PATIENT_DATA = {
    "gender": "male",
    "age": 31,
    "height_cm": 179,
    "weight_kg": 78,
    "skin_type": "sensitive",
    "allergies": [
        "nickel",
        "fragrances"
    ],
    "past_medical_history": [
        "eczema",
        "acne vulgaris",
        "psoriasis",
        "tinea pedis",
        "minor thermal burn on left forearm"
    ],
    "diabetes": "no",
    "current_medications": [
        "cetirizine as needed",
        "multivitamin"
    ],
    "chief_complaint": [
        "my hands have been very dry and ichy for months",
        "they get red and irritated and sometimes burn a little",
        "the skin peels and cracks and water makes it sting",
        "sometimes i see tiny bumps like little blisters",
        "it gets worse after frequent hand washing sanitizer and detergents",
        "moisturizer helps for a short time but then it comes back"
    ],
    "location_on_body": [
        "hands"
    ],
    "duration": "about 3 months",
    "symptoms": [
        "itching",
        "dryness",
        "redness",
        "peeling",
        "cracking pain",
        "occasional burning"
    ],
    "triggers": [
        "frequent hand washing",
        "hand sanitizer",
        "detergents"
    ]
}


In [22]:
# =========================
# 7) Convert patient dict to compact text
# =========================
def build_patient_info_text(patient_data: dict) -> str:
    def format_value(value):
        if isinstance(value, list):
            return ", ".join(str(v) for v in value)
        return str(value)

    ordered_keys = [
        "gender",
        "age",
        "height_cm",
        "weight_kg",
        "skin_type",
        "allergies",
        "past_medical_history",
        "diabetes",
        "current_medications",
        "chief_complaint",
        "location_on_body",
        "duration",
        "symptoms",
        "triggers",
    ]

    lines = []
    for key in ordered_keys:
        if key in patient_data and patient_data[key] not in [None, "", [], {}]:
            value = patient_data[key]
            if isinstance(value, list):
                if key == "chief_complaint" or key == "location_on_body":
                    lines.append(f"{key}:")
                    for item in value:
                        lines.append(f"- {item}")
                else:
                    lines.append(f"{key}: {format_value(value)}")
            else:
                lines.append(f"{key}: {format_value(value)}")

    return "\n".join(lines).strip()

PATIENT_INFO = build_patient_info_text(PATIENT_DATA)


In [23]:
# =========================
# 8) System instruction
# =========================
SYSTEM_INSTRUCTION = """
You are a conservative dermatology assistant used for doctor support.

Return ONLY plain English text.
Do NOT return JSON.
Do NOT return markdown.
Do NOT return bullet points.
Do NOT use headings.
Do NOT mention hidden reasoning.
Do NOT mention probabilities.
Do NOT mention that you corrected or summarized the complaint.

Before answering, silently do all of the following:
1) Correct obvious spelling mistakes in the patient's complaint.
2) Normalize and summarize the complaint into a short internal clinical impression.
3) Determine whether the uploaded image is actually relevant to dermatology.

Image relevance rules:
- If the uploaded content is not a dermatology-relevant skin image, return exactly:
  The most likely condition is undetected. The uploaded content does not appear to be a relevant dermatology image.
- Examples of irrelevant content include x-rays, CT scans, MRI scans, screenshots, documents, math problems, charts, random objects, or non-skin photos with no visible skin lesion.
- If the image is too blurry, too dark, too cropped, too distant, or otherwise not reliable for skin assessment, return exactly:
  The most likely condition is undetected. The image quality is not sufficient for a reliable dermatology assessment.

Diagnostic rules:
- Use the image as the primary evidence.
- Use patient metadata only as secondary supporting context.
- Do not invent visual findings.
- Do not rely mainly on past medical history.
- Past medical history may support recurrence only if the current image and present symptoms are also compatible with that condition.
- Ignore unrelated past conditions if they do not fit the current image and complaint.
- If the image clearly shows normal skin with no meaningful abnormality, you may say:
  The most likely condition is healthy skin.
- If the case is skin-related but too ambiguous for one reliable label, say:
  The most likely condition is undetermined.

Output format rules:
- Output must be 3 to 5 short sentences only.
- Sentence 1 must start exactly with:
  The most likely condition is ...
- Sentence 1 should use a broad umbrella diagnosis when appropriate, such as eczema, acne, fungal skin infection, psoriasis, or burn injury.
- Sentence 2 should, if reasonably supported by the image and history, provide a more specific clinical detail or subtype that may help the doctor.
- Sentence 3 must briefly explain the main reasons using visible findings and symptom history.
- Sentence 4 is optional and may mention whether past medical history supports recurrence or clinical context.
- Sentence 5 is optional and may contain brief low-risk practical advice.

Specificity rules:
- Prefer a broad umbrella label first, because the same output may also be shown to the patient.
- Add a more specific subtype only if it is reasonably supported by the current image and current complaint.
- Example:
  The most likely condition is eczema. More specifically, this appears most consistent with contact dermatitis.
- Do not force a subty# =========================
# 4) Model settings
# =========================
MODEL_NAME = "gemini-2.5-flash"
THINKING_BUDGET = 0
MAX_OUTPUT_TOKENS = 170
TEMPERATURE = 0.1pe if the subtype is uncertain.

Advice rules:
- Only include low-risk, general advice.
- Avoid herbal, botanical, essential-oil, or natural remedy recommendations.
- Do not prescribe drugs or doses.
- Safe advice may include avoiding triggers, reducing irritant exposure, using a gentle fragrance-free moisturizer, and seeking dermatology review if persistent or worsening.
""".strip()

In [24]:
# =========================
# 9) User prompt
# =========================
def build_prompt(patient_info_text: str) -> str:
    return f"""
Review ONE uploaded image together with the patient information below.

Task:
- Identify the single most likely skin condition.
- Use the image as the main evidence and the patient information as supporting context.
- If the upload is irrelevant to dermatology, return the exact undetected response.
- If the image is dermatology-related but still too uncertain for one reliable diagnosis, say:
  The most likely condition is undetermined.
- Start with a broad umbrella diagnosis that is easier for a patient to understand.
- Then, if supported, add a more specific clinical subtype or detail that may help the doctor.
- If a past condition such as eczema, acne, psoriasis, fungal infection, or burn history is relevant and consistent with the current image and symptoms, you may mention that it could support recurrence or clinical context.
- Do not focus only on history; the current image and current complaint must remain primary.
- Keep the answer concise and doctor-friendly.

Preferred style example:
The most likely condition is eczema. More specifically, this appears most consistent with contact dermatitis of the hands. This is suggested by redness, dryness, scaling, fissures, and worsening after frequent washing and detergent exposure. A past history of eczema may support recurrence. Advice: reduce irritant exposure, use a gentle fragrance-free moisturizer, and seek dermatology review if the rash persists or worsens.

Patient information:
{patient_info_text}
""".strip()

In [25]:
# =========================
# 10) Response cleaner
# =========================
def clean_response(text: str) -> str:
    if not text:
        return "The most likely condition is undetected. The image quality or content is not sufficient for a reliable dermatology assessment."

    s = text.strip()

    # Remove accidental code fences
    s = re.sub(r"^```[a-zA-Z0-9_-]*\n?", "", s)
    s = re.sub(r"```$", "", s).strip()

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # Keep only the first 5 sentences max
    parts = re.split(r'(?<=[.!?])\s+', s)
    parts = [p.strip() for p in parts if p.strip()]
    s = " ".join(parts[:5]).strip()

    if len(s) < 20:
        return "The most likely condition is undetected. The image quality or content is not sufficient for a reliable dermatology assessment."

    if not s.lower().startswith("the most likely condition is"):
        s = "The most likely condition is undetermined. " + s

    return s

In [27]:
# =========================
# 11) Main inference function
# =========================
def analyze_dermatology_case(
    image_bytes: bytes,
    mime_type: str,
    patient_data: dict,
    model_name: str = MODEL_NAME,
    thinking_budget: int = THINKING_BUDGET,
    max_output_tokens: int = MAX_OUTPUT_TOKENS,
    temperature: float = TEMPERATURE,
) -> str:
    patient_info_text = build_patient_info_text(patient_data)
    prompt = build_prompt(patient_info_text)

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type
    )

    response = client.models.generate_content(
        model=model_name,
        contents=[image_part, prompt],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            thinking_config=types.ThinkingConfig(
                thinking_budget=thinking_budget
            )
        )
    )

    return clean_response(response.text)

In [28]:
# =========================
# 12) Run the model
# =========================
final_response = analyze_dermatology_case(
    image_bytes=image_bytes,
    mime_type=mime_type,
    patient_data=PATIENT_DATA
)

print(final_response)


The most likely condition is eczema. More specifically, this appears most consistent with chronic hand eczema, possibly with a component of contact dermatitis. This is suggested by the red, scaly, and fissured plaques on the hands, along with symptoms of itching, dryness, and burning, which are exacerbated by frequent hand washing and detergents. A past history of eczema and allergies to nickel and fragrances supports this clinical context. Using a gentle, fragrance-free moisturizer and reducing exposure to irritants may help.


In [29]:
final_response

'The most likely condition is eczema. More specifically, this appears most consistent with chronic hand eczema, possibly with a component of contact dermatitis. This is suggested by the red, scaly, and fissured plaques on the hands, along with symptoms of itching, dryness, and burning, which are exacerbated by frequent hand washing and detergents. A past history of eczema and allergies to nickel and fragrances supports this clinical context. Using a gentle, fragrance-free moisturizer and reducing exposure to irritants may help.'

In [30]:
# =========================
# 13) Optional production-style usage example
# AWS side would pass image bytes + mime type + patient data
# =========================
# def handle_request_from_aws(image_bytes_from_aws, mime_type_from_aws, patient_data_from_aws):
#     result = analyze_dermatology_case(
#         image_bytes=image_bytes_from_aws,
#         mime_type=mime_type_from_aws,
#         patient_data=patient_data_from_aws
#     )
#     return result